# Rompecabezas 8-Puzzle: Aprendizaje por Refuerzo

## El juego
El 8-puzzle consiste en un tablero de 3×3 con fichas numeradas del **1 al 8** y un **espacio vacío (0)**.
El objetivo es **ordenar los números** deslizando fichas adyacentes al espacio vacío.

Estado objetivo:
```
1  2  3
4  5  6
7  8  (vacío)
```

In [ ]:
import numpy as np
import pickle
import random
import math

## 1. Entorno: Rompecabezas

In [ ]:
class Rompecabezas():
    def __init__(self):
        self.estado = np.array([[1, 2, 3],
                                [4, 5, 6],
                                [7, 8, 0]])

    def obtener_hueco(self):
        for i in range(3):
            for j in range(3):
                if self.estado[i, j] == 0:   
                    return (i, j)
        return None

    def movimientos_validos(self):
        fila_hueco, col_hueco = self.obtener_hueco()
        movimientos = []
        for df, dc in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
            nueva_fila = fila_hueco + df
            nueva_col = col_hueco + dc
            if 0 <= nueva_fila < 3 and 0 <= nueva_col < 3:
                movimientos.append((nueva_fila, nueva_col))
        return movimientos

    def mover(self, fila, col):
        fila_hueco, col_hueco = self.obtener_hueco()
        self.estado[fila_hueco, col_hueco] = self.estado[fila, col]
        self.estado[fila, col] = 0

    def esta_resuelto(self):
        objetivo = np.array([[1, 2, 3],
                            [4, 5, 6],
                            [7, 8, 0]])
        return np.array_equal(self.estado, objetivo)

    def mezclar(self, pasos=100):
        self.estado = np.array([[1, 2, 3],
                               [4, 5, 6],
                               [7, 8, 0]])
        for _ in range(pasos):
            movimientos = self.movimientos_validos()
            fila, col = random.choice(movimientos)
            self.mover(fila, col)

    def reiniciar(self):
        self.estado = np.array([[1, 2, 3],
                               [4, 5, 6],
                               [7, 8, 0]])

    def mostrar(self):
        for fila in self.estado:
            print(' '.join(str(int(x)) if x != 0 else ' ' for x in fila))

## 2. Agente

In [ ]:
class Agente():
    def __init__(self, tasa_aprendizaje=0.5, prob_exploracion=0.5):
        self.funcion_de_valor = {}
        self.tasa_aprendizaje = tasa_aprendizaje
        self.posiciones = []
        self.prob_exploracion = prob_exploracion

    def reiniciar(self):
        self.posiciones = []

    def mover(self, rompecabezas, explorar=True):
        movimientos = rompecabezas.movimientos_validos()
        if explorar and np.random.uniform(0, 1) < self.prob_exploracion:
            indice = np.random.choice(len(movimientos))
            return movimientos[indice]
        mejor_valor = -1000
        mejor_movimiento = movimientos[0]
        for fila, col in movimientos:
            siguiente_tablero = rompecabezas.estado.copy()
            fila_hueco, col_hueco = rompecabezas.obtener_hueco()
            siguiente_tablero[fila_hueco, col_hueco] = siguiente_tablero[fila, col]
            siguiente_tablero[fila, col] = 0
            estado_str = str(siguiente_tablero.reshape(9))
            valor = 0 if self.funcion_de_valor.get(estado_str) is None else self.funcion_de_valor.get(estado_str)
            if valor >= mejor_valor:
                mejor_valor = valor
                mejor_movimiento = (fila, col)
        return mejor_movimiento

    def actualizar(self, rompecabezas):
        estado_str = str(rompecabezas.estado.reshape(9))
        self.posiciones.append(estado_str)

    def recompensa(self, valor_recompensa):
        for estado in reversed(self.posiciones):
            if self.funcion_de_valor.get(estado) is None:
                self.funcion_de_valor[estado] = 0
            self.funcion_de_valor[estado] += self.tasa_aprendizaje * (valor_recompensa - self.funcion_de_valor[estado])
            valor_recompensa = self.funcion_de_valor[estado]

## 3. Juego

In [ ]:
from tqdm import tqdm

class Juego():
    def __init__(self, agente, max_pasos=100):
        self.agente = agente
        self.rompecabezas = Rompecabezas()
        self.max_pasos = max_pasos

    def jugar_episodio(self, pasos_mezcla=50):
        self.rompecabezas.mezclar(pasos_mezcla)
        self.agente.reiniciar()
        resuelto = False
        pasos = 0
        while not resuelto and pasos < self.max_pasos:
            pasos += 1
            accion = self.agente.mover(self.rompecabezas, explorar=True)
            self.rompecabezas.mover(accion[0], accion[1])
            self.agente.actualizar(self.rompecabezas)
            if self.rompecabezas.esta_resuelto():
                resuelto = True
        self.recompensa(resuelto)
        return resuelto, pasos

    def autoentrenar(self, episodios=10000):
        resultados = []
        for i in tqdm(range(1, episodios + 1)):
            resuelto, pasos = self.jugar_episodio()
            resultados.append((resuelto, pasos))
        return resultados

    def recompensa(self, resuelto):
        if resuelto:
            self.agente.recompensa(1)
        else:
            self.agente.recompensa(0)

---
# Exploración vs Explotación

Aplicamos la misma lógica del notebook `02_bandits.ipynb` al rompecabezas.

En lugar de elegir entre 5 topos con recompensas fijas, el agente debe elegir **qué movimiento hacer** en cada estado. Usamos los mismos métodos de acción-valor:
- **ε-greedy**: con probabilidad ε elige al azar
- **Implementación incremental**: actualización con tasa α constante
- **Valores iniciales optimistas**: V(s) inicial alto para explorar sin ε
- **UCB**: bonificación por exploración $c\sqrt{\ln(t)/N(s)}$
- **Gradiente (Softmax)**: preferencias con softmax

## Experimento 1: ε-greedy – Comparación de ε

Misma estructura que `02_bandits.ipynb`:
```
for partida in range(partidas):
    for i, eps in enumerate(epsilons):
        for turno in range(turnos):
            # jugar episodio
```

In [ ]:
np.random.seed(42)

partidas = 15
turnos = 3000
epsilons = [0, 0.05, 0.1, 0.4]

exitos_eps = np.zeros((len(epsilons), turnos))
pasos_eps = np.zeros((len(epsilons), turnos))

for partida in range(partidas):
    print(f"Partida {partida+1}/{partidas}", end=" ")
    for i, eps in enumerate(epsilons):
        agente = Agente(tasa_aprendizaje=0.3, prob_exploracion=eps)
        juego = Juego(agente, max_pasos=80)
        for turno in range(turnos):
            resuelto, pasos = juego.jugar_episodio()
            exitos_eps[i][turno] += (1 if resuelto else 0)
            pasos_eps[i][turno] += pasos
    print("-")

exitos_eps /= partidas
pasos_eps /= partidas

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
for i, eps in enumerate(epsilons):
    v = 50
    suave = np.convolve(exitos_eps[i], np.ones(v)/v, mode='valid')
    plt.plot(suave, label=f'ε = {eps}')
plt.legend()
plt.grid(True)
plt.xlabel('Episodios')
plt.ylabel('Tasa de éxito')
plt.title('ε-greedy: Tasa de éxito')

plt.subplot(1, 2, 2)
for i, eps in enumerate(epsilons):
    v = 50
    suave = np.convolve(pasos_eps[i], np.ones(v)/v, mode='valid')
    plt.plot(suave, label=f'ε = {eps}')
plt.legend()
plt.grid(True)
plt.xlabel('Episodios')
plt.ylabel('Pasos promedio')
plt.title('ε-greedy: Pasos por episodio')
plt.tight_layout()
plt.show()

**Interpretación:** ε=0 se estanca (nunca explora). ε mayor = más exploración, pero si es muy alto (0.4) aprende más lento.

## Experimento 2: Implementación incremental – Comparación de α

In [ ]:
partidas = 15
turnos = 3000
alphas = [0.05, 0.1, 0.3, 0.5, 0.9]
eps_fijo = 0.1

exitos_alpha = np.zeros((len(alphas), turnos))

for partida in range(partidas):
    print(f"Partida {partida+1}/{partidas}", end=" ")
    for i, alfa in enumerate(alphas):
        agente = Agente(tasa_aprendizaje=alfa, prob_exploracion=eps_fijo)
        juego = Juego(agente, max_pasos=80)
        for turno in range(turnos):
            resuelto, _ = juego.jugar_episodio()
            exitos_alpha[i][turno] += (1 if resuelto else 0)
    print("-")

exitos_alpha /= partidas

In [ ]:
plt.figure(figsize=(8, 4))
for i, alfa in enumerate(alphas):
    v = 50
    suave = np.convolve(exitos_alpha[i], np.ones(v)/v, mode='valid')
    plt.plot(suave, label=f'α = {alfa}')
plt.legend()
plt.grid(True)
plt.xlabel('Episodios')
plt.ylabel('Tasa de éxito')
plt.title(f'Efecto de α (ε={eps_fijo})')
plt.show()

**Interpretación:** α bajo (0.05) aprende muy lento. α alto (0.9) es inestable. α intermedio (0.1–0.3) da mejor balance.

## Experimento 3: Valores iniciales optimistas

Inicializamos V(s) con un valor alto (5) para que incluso un agente greedy (ε=0) explore todas las opciones antes de conformarse.

In [ ]:
from collections import defaultdict

partidas = 10
turnos = 3000

metodos_opt = ['Greedy (ε=0, V0=0)', 'Greedy optimista (ε=0, V0=5)', 'ε-greedy (ε=0.1)']
exitos_opt = np.zeros((len(metodos_opt), turnos))

for partida in range(partidas):
    print(f"Partida {partida+1}/{partidas}", end=" ")

    agente0 = Agente(tasa_aprendizaje=0.3, prob_exploracion=0)
    juego0 = Juego(agente0, max_pasos=80)
    for turno in range(turnos):
        resuelto, _ = juego0.jugar_episodio()
        exitos_opt[0][turno] += (1 if resuelto else 0)

    agente1 = Agente(tasa_aprendizaje=0.3, prob_exploracion=0)
    agente1.funcion_de_valor = defaultdict(lambda: 5)
    juego1 = Juego(agente1, max_pasos=80)
    for turno in range(turnos):
        resuelto, _ = juego1.jugar_episodio()
        exitos_opt[1][turno] += (1 if resuelto else 0)

    agente2 = Agente(tasa_aprendizaje=0.3, prob_exploracion=0.1)
    juego2 = Juego(agente2, max_pasos=80)
    for turno in range(turnos):
        resuelto, _ = juego2.jugar_episodio()
        exitos_opt[2][turno] += (1 if resuelto else 0)

    print("-")

exitos_opt /= partidas

In [ ]:
plt.figure(figsize=(8, 4))
for i, m in enumerate(metodos_opt):
    v = 50
    suave = np.convolve(exitos_opt[i], np.ones(v)/v, mode='valid')
    plt.plot(suave, label=m)
plt.legend()
plt.grid(True)
plt.xlabel('Episodios')
plt.ylabel('Tasa de éxito')
plt.title('Valores iniciales optimistas')
plt.show()

**Interpretación:** El greedy optimista (V0=5) explora forzadamente porque cada estado nuevo tiene valor 5, que baja al visitarlo, empujando al agente a probar otros estados.

## Experimento 4: UCB (Upper Confidence Bound)

Selecciona acciones maximizando $Q(s) + c\sqrt{\ln(t)/N(s)}$. Implementado externamente (sin modificar clases).

In [ ]:
partidas = 10
turnos = 2000
valores_c = [0.5, 1.0, 2.0]

nombres_ucb = [f'UCB (c={c})' for c in valores_c] + ['ε-greedy (ε=0.1)']
exitos_ucb = np.zeros((len(nombres_ucb), turnos))

for partida in range(partidas):
    print(f"Partida {partida+1}/{partidas}", end=" ")

    for i, c in enumerate(valores_c):
        V = {}
        N = {}
        puzzle = Rompecabezas()
        paso_global = 0
        for turno in range(turnos):
            puzzle.mezclar(50)
            resuelto = False
            historial = []
            for paso in range(80):
                paso_global += 1
                estado_act = str(puzzle.estado.reshape(9))
                if N.get(estado_act) is None:
                    N[estado_act] = 0
                N[estado_act] += 1
                movs = puzzle.movimientos_validos()
                mejor_p = -1e9
                mejor_m = movs[0]
                for fila, col in movs:
                    sig = puzzle.estado.copy()
                    fh, ch = puzzle.obtener_hueco()
                    sig[fh, ch] = sig[fila, col]
                    sig[fila, col] = 0
                    e_sig = str(sig.reshape(9))
                    valor = 0 if V.get(e_sig) is None else V.get(e_sig)
                    n = 1 if N.get(e_sig) is None else N.get(e_sig)
                    puntaje = valor + c * math.sqrt(math.log(paso_global + 1) / n)
                    if puntaje > mejor_p:
                        mejor_p = puntaje
                        mejor_m = (fila, col)
                puzzle.mover(mejor_m[0], mejor_m[1])
                historial.append(estado_act)
                if puzzle.esta_resuelto():
                    resuelto = True
                    break
            r = 1 if resuelto else 0
            for est in reversed(historial):
                if V.get(est) is None:
                    V[est] = 0
                V[est] += 0.3 * (r - V[est])
                r = V[est]
            exitos_ucb[i][turno] += (1 if resuelto else 0)

    agente = Agente(tasa_aprendizaje=0.3, prob_exploracion=0.1)
    juego = Juego(agente, max_pasos=80)
    for turno in range(turnos):
        resuelto, _ = juego.jugar_episodio()
        exitos_ucb[len(valores_c)][turno] += (1 if resuelto else 0)

    print("-")

exitos_ucb /= partidas

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
for i, nombre in enumerate(nombres_ucb):
    v = 50
    suave = np.convolve(exitos_ucb[i], np.ones(v)/v, mode='valid')
    plt.plot(suave, label=nombre)
plt.legend()
plt.grid(True)
plt.xlabel('Episodios')
plt.ylabel('Tasa de éxito')
plt.title('UCB vs ε-greedy')

plt.subplot(1, 2, 2)
for i, nombre in enumerate(nombres_ucb):
    final = np.mean(exitos_ucb[i][-500:])
    plt.bar(i, final)
plt.xticks(range(len(nombres_ucb)), nombres_ucb, rotation=45)
plt.ylabel('Tasa de éxito final')
plt.title('Comparación final')
plt.tight_layout()
plt.show()

**Interpretación:** UCB explora de forma más inteligente que ε-greedy, priorizando acciones con alta incertidumbre.

## Experimento 5: Algoritmo de Gradiente (Softmax)

Asigna preferencias $H(a)$ a cada acción y las convierte en probabilidades con softmax. Actualiza por gradiente comparando la recompensa con el promedio.

In [ ]:
def softmax(x):
    return np.exp(x) / sum(np.exp(x))

In [ ]:
partidas = 10
turnos = 2000
alphas_grad = [0.05, 0.1, 0.3]

nombres_grad = [f'Gradiente (α={a})' for a in alphas_grad] + ['ε-greedy (ε=0.1)']
exitos_grad = np.zeros((len(nombres_grad), turnos))

for partida in range(partidas):
    print(f"Partida {partida+1}/{partidas}", end=" ")

    for i, alfa in enumerate(alphas_grad):
        H = {}
        puzzle = Rompecabezas()
        historial_recompensas = []
        for turno in range(turnos):
            puzzle.mezclar(50)
            resuelto = False
            historial = []
            for paso in range(80):
                estado = str(puzzle.estado.reshape(9))
                movs = puzzle.movimientos_validos()
                if H.get(estado) is None:
                    H[estado] = np.zeros(len(movs))
                probs = softmax(H[estado])
                idx = np.random.choice(len(movs), p=probs)
                mov = movs[idx]
                puzzle.mover(mov[0], mov[1])
                historial.append((estado, idx, len(movs)))
                if puzzle.esta_resuelto():
                    resuelto = True
                    break
            r = 1 if resuelto else 0
            historial_recompensas.append(r)
            r_prom = np.mean(historial_recompensas)
            for estado, idx_acc, n_acc in reversed(historial):
                probs = softmax(H[estado])
                H[estado][idx_acc] += alfa * (r - r_prom) * (1 - probs[idx_acc])
                for j in range(n_acc):
                    if j != idx_acc:
                        H[estado][j] -= alfa * (r - r_prom) * probs[j]
                r = 0
            exitos_grad[i][turno] += (1 if resuelto else 0)

    agente = Agente(tasa_aprendizaje=0.3, prob_exploracion=0.1)
    juego = Juego(agente, max_pasos=80)
    for turno in range(turnos):
        resuelto, _ = juego.jugar_episodio()
        exitos_grad[len(alphas_grad)][turno] += (1 if resuelto else 0)

    print("-")

exitos_grad /= partidas

In [ ]:
plt.figure(figsize=(8, 4))
for i, nombre in enumerate(nombres_grad):
    v = 50
    suave = np.convolve(exitos_grad[i], np.ones(v)/v, mode='valid')
    plt.plot(suave, label=nombre)
plt.legend()
plt.grid(True)
plt.xlabel('Episodios')
plt.ylabel('Tasa de éxito')
plt.title('Gradiente (Softmax) vs ε-greedy')
plt.show()

---
## Resumen

| Método | Idea | Cuándo usarlo |
|---|---|---|
| **ε-greedy** | Con prob ε elige al azar | Simple, funciona en general |
| **Incremental (α)** | Actualizar con tasa α fija | No guardar todas las recompensas |
| **Valores optimistas** | V(s) inicial alto | Forzar exploración sin ε |
| **UCB** | Bonus $c\sqrt{\ln(t)/N(s)}$ | Exploración más inteligente |
| **Gradiente** | Softmax + preferencias | Exploración probabilística natural |

Sin modificar `Rompecabezas`, `Agente` ni `Juego`. Sin crear nuevas clases.